In [1]:
import torchvision
import torch
from PIL import Image
import torch.nn as nn
from torch import optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import random

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [3]:
# Replace last classifier to only handle 2 cases, one benign one high grade
num_classes = 2
model = torchvision.models.alexnet(pretrained = True)
# replace the last classifier
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[6].parameters():
    param.requires_grad = True

In [5]:
labels = pd.read_csv('/content/case_grade_match.csv')

In [6]:
# Need to group the patches by their cases, and also need to randomly split, the function below will group the cases
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        # If the image size is less than a kilobyte, don't use the patch
        if os.path.getsize(os.path.join(patch_dir, filename)) < 1000:
            continue
        # If 'patched' in filename, not an actual patch
        if 'patched' in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        # Get patches, if they are not patch, don't use the patch
        image = Image.open(image_path).convert('RGB')
        # Get the label information using the labels dataframe based on case number
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

# Group the patches
patches = group_patches('/content/drive/MyDrive/Patches/')

# Get case numbers and their labels
case_nums = list(patches.keys())
dataset = labels.loc[[(int(x)-1) for x in case_nums]]
# Remove those that are equal to 2
noindex = dataset.Class != 2.0
X = dataset[noindex].Case
y = dataset[noindex].Class
train, test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=40)

# Create the training patches and the test patches
train_patches = {case_num: patches[int(case_num)] for case_num in train}
test_patches = {case_num: patches[int(case_num)] for case_num in test}


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Generate the actual datasets for the models
train_dataset = PNGDataset(train_patches, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform = transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [19]:
type(train_dataset)

__main__.PNGDataset

In [9]:
os.path.exists('/content/drive/MyDrive/Patches/case_82_match_1_sox10_patch213.png')

True

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5  # Number of epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)
        loss = criterion(outputs, labels)


        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_dataloader):.4f}')

/content/drive/MyDrive/Patches/case_059_match_1_sox10_patch455.png
/content/drive/MyDrive/Patches/case_030_match_1_melan_patch15.png
/content/drive/MyDrive/Patches/case_094_match_1_melan_patch39.png
/content/drive/MyDrive/Patches/case_091_match_2_melan_patch0.png
/content/drive/MyDrive/Patches/case_64_match_1_sox10_patch257.png
/content/drive/MyDrive/Patches/case_87_match_1_h&e_patch293.png
/content/drive/MyDrive/Patches/case_89_match_1_h&e_patch133.png
/content/drive/MyDrive/Patches/case_79_match_2_sox10_patch68.png


FileNotFoundError: [Errno 2] No such file or directory

In [ ]:
correct = 0
total = 0
model.eval()  # Set model to evaluation mode
with torch.no_grad():  # Disable gradient computation
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)  # Get the predicted class
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy:.2f}%')